In [5]:
import requests
import wikipedia
from docx import Document
import re
import textwrap

# =========================
# LM STUDIO
# =========================
LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"

chat_history = []

def ask_llama(prompt):
    try:
        messages = chat_history + [{"role": "user", "content": prompt}]

        response = requests.post(
            LM_STUDIO_URL,
            json={
                "model": "llama",
                "messages": messages,
                "temperature": 0.7
            }
        )

        reply = response.json()['choices'][0]['message']['content']

        chat_history.append({"role": "user", "content": prompt})
        chat_history.append({"role": "assistant", "content": reply})

        return reply

    except Exception as e:
        return f"LM Studio Error: {e}"

# =========================
# TOOLS
# =========================
def is_math(text):
    return bool(re.fullmatch(r"[0-9+\-*/(). ]+", text))

def calculator(expr):
    try:
        return eval(expr)
    except:
        return "Invalid math expression"

def wiki_search(query):
    try:
        return wikipedia.summary(query, sentences=3)
    except:
        return None  # fallback handled later

def generate_word_doc(content, filename="output.docx"):
    try:
        doc = Document()

        # Title
        doc.add_heading("Generated Document", 0)

        # Add paragraphs
        for para in content.split("\n"):
            if para.strip():
                doc.add_paragraph(para)

        doc.save(filename)
        return f"Document saved as {filename}"

    except Exception as e:
        return f"Document Error: {e}"

def print_long(text):
    print(textwrap.fill(text, width=100))


# =========================
# CHAT LOOP
# =========================
print("AI Chat Started (type 'exit' to stop)\n")

while True:

    user_input = input("User: ")
    print(f"\nUser Input: {user_input}\n")

    if user_input.lower() == "exit":
        print("AI: Goodbye!")
        break

    # =========================
    # 1. MATH
    # =========================
    if is_math(user_input):
        print("AI:", calculator(user_input))

    # =========================
    # 2. WIKIPEDIA
    # =========================
    elif any(word in user_input.lower() for word in ["what is", "who is", "define", "meaning"]):

        wiki_result = wiki_search(user_input)

        if wiki_result:
            print("\nAI (Wikipedia):")
            print_long(wiki_result)
        else:
            print("\nAI:")
            print_long(ask_llama(user_input))

    # =========================
    # 3. WORD DOCUMENT GENERATOR (FIXED ✅)
    # =========================
    elif "document" in user_input.lower():

        print("\nAI: Generating document...\n")

        # Clean the prompt so the doc doesn't include "create document"
        clean_prompt = user_input.lower() \
            .replace("create", "") \
            .replace("make", "") \
            .replace("generate", "") \
            .replace("document", "") \
            .strip()

        if not clean_prompt:
            clean_prompt = "Write a general informative document."

        # Generate content using AI
        content = ask_llama(clean_prompt)

        # Save to Word file
        result = generate_word_doc(content)

        print(result)

    # =========================
    # 4. DEFAULT CHAT AI
    # =========================
    else:
        print("\nAI:")
        print_long(ask_llama(user_input))

    print("\n" + "-"*60 + "\n")

AI Chat Started (type 'exit' to stop)


User Input: create me a document about ai


AI: Generating document...

Document saved as output.docx

------------------------------------------------------------


User Input: 1+1

AI: 2

------------------------------------------------------------


User Input: 2/2

AI: 1.0

------------------------------------------------------------


User Input: define arduino


AI (Wikipedia):
Arduino () is an Italian open-source hardware and software company (now owned by Qualcomm), as well
as a project and user community that designs and manufactures single-board microcontrollers and
microcontroller kits for building digital and other kinds of devices. Its hardware products are
licensed under a CC BY-SA license, while the software is licensed under the GNU Lesser General
Public License (LGPL) or the GNU General Public License (GPL), permitting the manufacture of Arduino
boards and software distribution by anyone. Arduino boards are available commercially